### CS 180 Machine Exercise 1 (ME1): SMS Spam Classification with Naive Bayes

**Objective**

Build and hypertune a Naive Bayes classifier to detect spam messages using the SMS Spam Collection dataset. The model should achieve ≥96 % accuracy on the test set.

**Dataset**
- Use the SMS Spam Collection Dataset (available on UCI Machine Learning Repository (archive.ics.uci.edu)).
- It contains 5,574 SMS messages labeled as ham (legitimate) or spam.


**Steps**

**1. Load and Explore the Dataset** (10 points)
- Import the dataset into a pandas DataFrame.
- Inspect class distribution (ham vs spam).
- Perform basic text preprocessing (lowercasing, removing punctuation, optional stopword removal).


In [202]:
#Place code here
import pandas as pd
import csv
import string
import re

df = pd.read_csv(
  "sms+spam+collection/SMSSpamCollection",
  sep='\t',
  header=None,
  names=["label", "text"],
  quoting=csv.QUOTE_NONE
)

# Ensure that labels are parsed correctly
assert set(df["label"].unique()) == {"ham", "spam"}

# Isnpect dataset
print(df.head(), "\n")
df.info()

# Verify dataset size
num_rows, num_cols = df.shape
assert num_rows == 5_574 and num_cols == 2

# Inspect class distribution
print("\nClass distribution:")
print(df["label"].value_counts()) # ham: 4827, spam: 747

print("\nClass distribution (%):")
print(df["label"].value_counts(normalize=True) * 100) # ham: approx 87%, spam: approx 13%

# Basic text preprocessing
def preprocess_text(text: str):
  # lowercasing
  text = text.lower()

  # removing punctuation
  translator = str.maketrans('', '', string.punctuation)
  text = text.translate(translator)

  # remove unnecessary white space(s)
  text = re.sub(r"\s+", " ", text).strip()

  return text

df["processed"] = df["text"].apply(preprocess_text)

# Sanity check
print("\nPreprocessing sanity check:")
print(f"Original: {df["text"].iloc[67]}")
print(f"Processed: {df["processed"].iloc[67]}")

  label                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro... 

<class 'pandas.DataFrame'>
RangeIndex: 5574 entries, 0 to 5573
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   label   5574 non-null   str  
 1   text    5574 non-null   str  
dtypes: str(2)
memory usage: 87.2 KB

Class distribution:
label
ham     4827
spam     747
Name: count, dtype: int64

Class distribution (%):
label
ham     86.598493
spam    13.401507
Name: proportion, dtype: float64

Preprocessing sanity check:
Original: Urgent UR awarded a complimentary trip to EuroDisinc Trav, Aco&Entry41 Or £1000. To claim txt DIS to 87121 18+6*£1.50(moreFrmMob. ShrAcomOrSglSuplt)10, LS

**2. Feature Extraction** (20 points)
- Use scikit-learn’s CountVectorizer or TfidfVectorizer to convert text into numerical features.
- Experiment with:
- ngram_range (e.g., (1,1), (1,2))
- max_features (e.g., 5000, 10000)
- stop_words (e.g., "english")


In [203]:
# Place code here
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_combos = [
    {"name": "unigram 5000", "ngram_range": (1,1), "max_features": 5000, "stop_words": None},
    {"name": "unigram,bigram 5000", "ngram_range": (1,2), "max_features": 5000, "stop_words": None},
    {"name": "unigram,bigram 5000 + stopwords", "ngram_range": (1,2), "max_features": 5000, "stop_words": "english"},
    {"name": "unigram,bigram 10000 + stopwords", "ngram_range": (1,2), "max_features": 10000, "stop_words": "english"}
]

res = []

for config in vectorizer_combos:
    print(f"Experiment: {config['name']}")

    vectorizer = TfidfVectorizer(
        ngram_range=config["ngram_range"],
        max_features=config["max_features"],
        stop_words=config["stop_words"]
    )

    X = vectorizer.fit_transform(df['processed'])

    print(f"Feature matrix shape: {X.shape}")
    print(f"First 10 feature names: {vectorizer.get_feature_names_out()[:10]}\n")

    res.append((config["name"], X.shape, X))

# Summary of experiments
print("Summary of TF-IDF feature extraction experiments:")
for name, shape, _ in res:
    print(f"{name}: {shape}")


Experiment: unigram 5000
Feature matrix shape: (5574, 5000)
First 10 feature names: ['008704050406' '01223585236' '01223585334' '0125698789' '02' '020603'
 '0207' '02070836089' '02072069400' '02073162414']

Experiment: unigram,bigram 5000
Feature matrix shape: (5574, 5000)
First 10 feature names: ['020603 this' '0800' '08000839402' '08000839402 or' '08000930705'
 '08000930705 for' '08001950382' '08001950382 or' '08002986906' '0808']

Experiment: unigram,bigram 5000 + stopwords
Feature matrix shape: (5574, 5000)
First 10 feature names: ['008704050406 sp' '01223585334' '01223585334 cum' '020603' '020603 2nd'
 '0207' '02073162414' '02073162414 costs' '020903' '020903 2nd']

Experiment: unigram,bigram 10000 + stopwords
Feature matrix shape: (5574, 10000)
First 10 feature names: ['008704050406' '008704050406 sp' '0089my digits' '0121' '0121 2025050'
 '01223585236' '01223585236 xx' '01223585334' '01223585334 cum'
 '0125698789']

Summary of TF-IDF feature extraction experiments:
unigram 5000:

**3. Train-Test Split** (5 points)
- Split dataset into 80% training, 20% testing using train_test_split.


In [204]:
# Put code here
from sklearn.model_selection import train_test_split

X = res[2][2]
y = df["label"].map({"ham": 0, "spam": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)

# sanity check
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")

X_train: (4459, 5000), y_train: (4459,)
X_test: (1115, 5000), y_test: (1115,)


**4. Model Training** (40 points)

- Use Multinomial Naive Bayes (MultinomialNB) from scikit-learn.
- Hypertune parameters:
- alpha (smoothing parameter, try values like 0.1, 0.5, 1.0)
- Feature extraction parameters (from step 2).


In [205]:
# Put code here
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import GridSearchCV

param_grid = {'alpha':[0.01, 0.1, 0.5, 1.0, 2.0]}

baseline_model = MultinomialNB().fit(X_train,y_train)
baseline_preds = baseline_model.predict(X_test)

grid = GridSearchCV(MultinomialNB(),param_grid,cv=5)
grid.fit(X_train,y_train)

print(f"Best parameters: {grid.best_params_}")
print(f"Best cross-validation score: {grid.best_score_:.3f}")

tuned_model = grid.best_estimator_
tuned_preds = tuned_model.predict(X_test)


Best parameters: {'alpha': 0.1}
Best cross-validation score: 0.981


**5. Evaluation Metrics** (20 points)
- Measure accuracy, precision, recall, and F1-score.
- Ensure accuracy ≥95% on the test set.


In [206]:
# Put code here
from sklearn.metrics import classification_report

print(classification_report(y_test, tuned_preds))


              precision    recall  f1-score   support

           0       0.99      1.00      0.99       954
           1       0.99      0.91      0.95       161

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115



**6. Reflection & Insights** (5 points)
- Explains which preprocessing and parameter choices improved performance


Explanation:
Upon manually checking the experiments, a 1% improvement was brought about by:
- Bigrams: Including this captured important words pairs for ham/spam classification
- Stopwords removal: This improved accuracy likely because the remaining words were more informative for spam detection
- 5000 features: Limiting to this amount might have reduced noise from rarer words
- alpha value of 0.1: This improved propability estimates and accuracy